In [9]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import xarray as xr
import os
import re
import importlib as il
from scipy.optimize import curve_fit

from tqdm import tqdm
from datetime import datetime, timedelta
import time
import sys
sys.path.append('/glade/work/prasoonv/hpc_course/hw4/')
sys.path.append('/glade/work/prasoonv/hpc_course/hw4/src/')


# NumPY Implementation

In [3]:
import lorentz_func

# Baseline pipeline: generate samples, histogram, visualize, and run correctness checks
n = 10**8
bins = 100
xmin, xmax = -10, 10

start = time.perf_counter()
counts = lorentz_func.lorentzian_histogram(n, bins)
end = time.perf_counter() - start
print(f"Baseline Lorentzian histogram: {end:.2f} seconds")
bin_width = (xmax - xmin) / bins

# Normalized histogram estimate of p(x)
density = counts / (n * bin_width)
bin_centers = np.linspace(xmin + bin_width/2, xmax - bin_width/2, bins)

# Theoretical Lorentzian PDF
x_line = np.linspace(xmin, xmax, 1000)
pdf = 1.0 / (np.pi * (1.0 + x_line**2))

fig = go.Figure()
fig.add_bar(x=bin_centers, y=density, name="Normalized histogram", opacity=0.6)
fig.add_scatter(x=x_line, y=pdf, mode="lines", name="Lorentzian PDF", line=dict(width=3))
fig.update_layout(
    title="Lorentzian samples: normalized histogram vs theoretical PDF",
    xaxis_title="x",
    yaxis_title="PDF p(x)",
    template="plotly_dark"
)
fig.show()

Baseline Lorentzian histogram: 3.20 seconds


In [11]:

# Correctness checks

# (i) Check 1
x_sorted = np.sort(x)
F_theory = 0.5 + np.arctan(x_sorted) / np.pi
i = np.arange(1, n + 1)
D_plus = np.max(i / n - F_theory)
D_minus = np.max(F_theory - (i - 1) / n)
D_lorentz = max(D_plus, D_minus)

# (ii) Check 2
pvals = np.array([0.25, 0.50, 0.75])
sample_q = np.quantile(x, pvals)
theory_q = np.tan(np.pi * (pvals - 0.5))

print(f"Check 1: KS statistic vs Lorentzian(0,1): D = {D_lorentz:.5f}\n")
print("Check 2: Quantiles of sample vs theory:")
for p, sq, tq in zip(pvals, sample_q, theory_q):
    print(f"  p={p:.2f}: sample={sq:.5f}, theory={tq:.5f}, diff={sq - tq:+.5f}")


Check 1: KS statistic vs Lorentzian(0,1): D = 0.00013

Check 2: Quantiles of sample vs theory:
  p=0.25: sample=-0.99973, theory=-1.00000, diff=+0.00027
  p=0.50: sample=0.00026, theory=0.00000, diff=+0.00026
  p=0.75: sample=1.00070, theory=1.00000, diff=+0.00070


In [6]:
def strong_scaling(func, n = 10**7, bins = 100, technique=None):
    
    times = []
    parallel_count = [1, 2, 4, 8, 16, 32]
    
    for p in tqdm(parallel_count):
        
        start = time.perf_counter()
        func(n, n_counts = p, bins = bins)
        end = time.perf_counter() - start
        times.append(end)
    
    fig = go.Figure()
    fig.add_scatter(x=parallel_count, y=times, opacity=0.6)
    fig.update_layout(
        title=f"Strong Scaling: {technique} for n={n:.0e}, bins={bins}",
        xaxis_title="Parallel Divisions",
        yaxis_title="Time (seconds)",
        template="plotly_dark"
    )
    fig.update_xaxes(
        tickvals=parallel_count,
    )
    fig.show()
    fig.write_html(f"{technique}_strong_scaling.html")
    
    return times


In [5]:

def weak_scaling(func, n_per_div = 10**7, bins = 100, technique=None):
    
    times = []
    parallel_count = [1, 2, 4, 8]
    
    for p in tqdm(parallel_count):
        
        n = n_per_div * p
        
        start = time.perf_counter()
        func(n, n_counts = p, bins = bins)
        end = time.perf_counter() - start
        times.append(end)
    
    fig = go.Figure()
    fig.add_scatter(x=parallel_count, y=times, opacity=0.6)
    fig.update_layout(
        title=f"Weak Scaling: {technique} for n per divisions={n_per_div:.0e}, bins={bins}",
        xaxis_title="Parallel Divisions",
        yaxis_title="Time (seconds)",
        template="plotly_dark",
    )
    fig.update_xaxes(
        tickvals=parallel_count,
    )
    fig.show()
    fig.write_html(f"{technique}_weak_scaling.html")
    
    return times

In [7]:
def speedup(times):
    speedups = times[0] / np.array(times)
    
    fig = go.Figure()
    fig.add_scatter(x=[1, 2, 4, 8, 16, 32][:len(times)], y=speedups, opacity=0.6)
    fig.update_layout(
        title=f"Speedup vs Parallel Divisions",
        xaxis_title="Parallel Divisions",
        yaxis_title="Speedup",
        template="plotly_dark"
    )
    fig.update_xaxes(
        tickvals=[1, 2, 4, 8, 16, 32][:len(times)],
    )
    #fig.show()
    return speedups
    
def efficiency(speedups, parallel_count):
    efficiencies = speedups / np.array(parallel_count)
    
    fig = go.Figure()
    fig.add_scatter(x=parallel_count[:len(efficiencies)], y=efficiencies, opacity=0.6)
    fig.update_layout(
        title=f"Efficiency vs Parallel Divisions",
        xaxis_title="Parallel Divisions",
        yaxis_title="Efficiency",
        template="plotly_dark"
    )
    fig.update_xaxes(
        tickvals=parallel_count[:len(efficiencies)],
    )
    fig.show()
    return efficiencies

In [37]:
def plot_scaling(counts, technique, n=10**7, bins=100):

    xmin, xmax = -10, 10
    bin_width = (xmax - xmin) / bins
    
    # Normalized histogram estimate of p(x)
    density = counts / (n*bin_width)
    bin_centers = np.linspace(xmin + bin_width/2, xmax - bin_width/2, bins)

    # Theoretical Lorentzian PDF
    x_line = np.linspace(xmin, xmax, 1000)
    pdf = 1.0 / (np.pi * (1.0 + x_line**2))

    fig = go.Figure()
    fig.add_bar(x=bin_centers, y=density, name="Normalized histogram", opacity=0.6)
    fig.add_scatter(x=x_line, y=pdf, mode="lines", name="Lorentzian PDF", line=dict(width=3))

    fig.update_layout(
        title=f"Lorentzian samples: normalized histogram using {technique}",
        xaxis_title="x",
        yaxis_title="PDF p(x)",
        template="plotly_dark"
    )
    fig.show()


# Parallelization Approaches

### (a) Threading

In [16]:
import threading
import thread_lorentz
import time
import importlib as il

In [32]:
thread_lorentz = il.reload(thread_lorentz)

n = 10**7
bins = 1000

t = [1,2,4,8,16,32]


thread_strong_times = strong_scaling(thread_lorentz.run_threaded, n=n, bins=bins, technique='Threading')
thread_weak_times = weak_scaling(thread_lorentz.run_threaded, n_per_div=n, bins=bins, technique='Threading')

np.savetxt("threading_strong.txt", thread_strong_times)
np.savetxt("threading_weak.txt", thread_weak_times)

speedups = speedup(thread_strong_times)
efficiencies = efficiency(speedups,t)

                                     

100%|██████████| 6/6 [00:00<00:00,  6.43it/s]


100%|██████████| 4/4 [00:11<00:00,  3.00s/it]


In [39]:
n = 10**7
bins = 100

counts = thread_lorentz.run_threaded(n, n_counts=32, bins=bins)

plot_scaling(counts, technique="Threading", n=n, bins=bins)

In [9]:
os.cpu_count()

128

### (b) Multiprocessing

In [40]:
import mp_lorentz
mp_lorentz = il.reload(mp_lorentz)

from mp_lorentz import run_multiproc

In [9]:

n = 10**7
bins = 1000
t = [1,2,4,8,16,32]

mp_strong_times = strong_scaling(run_multiproc, n=n, bins=bins, technique='Multiprocessing')
mp_weak_times = weak_scaling(run_multiproc, n_per_div=n, bins=bins, technique='Multiprocessing')

np.savetxt("multiprocessing_strong.txt", mp_strong_times)
np.savetxt("multiprocessing_weak.txt", mp_weak_times)

speedups = speedup(mp_strong_times)
efficiencies = efficiency(speedups,t)

100%|██████████| 6/6 [00:01<00:00,  4.18it/s]


100%|██████████| 4/4 [00:03<00:00,  1.02it/s]


In [42]:
n = 10**7
bins = 100

counts = run_multiproc(n, n_counts=32, bins=bins)

plot_scaling(counts, technique="Multiprocessing", n=n, bins=bins)

### (c) ProcessPoolExecutor

In [43]:
import ppe_lorentz
from concurrent.futures import ProcessPoolExecutor
from ppe_lorentz import run_ppe


In [12]:

n = 10**7
bins = 1000

pp_strong_times = strong_scaling(run_ppe, n=n, bins=bins, technique='ProcessPoolExecutor')
pp_weak_times = weak_scaling(run_ppe, n_per_div=n, bins=bins, technique='ProcessPoolExecutor')

np.savetxt("ppe_strong.txt", pp_strong_times)
np.savetxt("ppe_weak.txt", pp_weak_times)

speedups = speedup(pp_strong_times)
efficiencies = efficiency(speedups,t)


100%|██████████| 6/6 [00:01<00:00,  4.25it/s]


100%|██████████| 4/4 [00:01<00:00,  2.34it/s]


In [45]:
n = 10**7
bins = 100

counts = run_ppe(n, n_counts=32, bins=bins)

plot_scaling(counts, technique="ProcessPoolExecutor", n=n, bins=bins)

### (d) Asyncio

In [55]:
import async_lorentz
async_lorentz = il.reload(async_lorentz)
from async_lorentz import run_async


In [14]:

n = 10**7
bins = 1000

as_strong_times = strong_scaling(run_async, n=n, bins=bins, technique='Asyncio')
as_weak_times = weak_scaling(run_async, n_per_div=n, bins=bins, technique='Asyncio')

np.savetxt("async_strong.txt", as_strong_times)
np.savetxt("async_weak.txt", as_weak_times)

speedups = speedup(as_strong_times)
efficiencies = efficiency(speedups,t)

100%|██████████| 6/6 [00:01<00:00,  3.34it/s]


100%|██████████| 4/4 [00:04<00:00,  1.17s/it]


In [62]:
n = 10**7
bins = 100

counts = async_lorentz.get_counts(n, n_tasks=32, bins=bins)

counts = await counts
plot_scaling(counts, technique="Asyncio", n=n, bins=bins)

### (e) Dask

In [68]:
import dask
import dask_lorentz
dask_lorentz = il.reload(dask_lorentz)
from dask_lorentz import run_dask

In [16]:

n = 10**7
bins = 1000

da_strong_times = strong_scaling(run_dask, n=n, bins=bins, technique='Dask')
da_weak_times = weak_scaling(run_dask, n_per_div=n, bins=bins, technique='Dask')

np.savetxt("dask_strong.txt", da_strong_times)
np.savetxt("dask_weak.txt", da_weak_times)

speedups = speedup(da_strong_times)
efficiencies = efficiency(speedups,t)

100%|██████████| 6/6 [00:01<00:00,  5.98it/s]


100%|██████████| 4/4 [00:01<00:00,  2.03it/s]


In [69]:
n = 10**7
bins = 100

counts = run_dask(n=n, n_counts=32, bins=bins)
plot_scaling(counts, technique="Dask", n=n, bins=bins)

### (f) Numba

In [71]:
import numba_lorentz
numbar_lorentz = il.reload(numba_lorentz)
from numba_lorentz import run_numba

In [19]:

n = 10**7
bins = 1000

nu_strong_times = strong_scaling(run_numba, n=n, bins=bins, technique='Numba')
nu_weak_times = weak_scaling(run_numba, n_per_div=n, bins=bins, technique='Numba')

np.savetxt("numba_strong.txt", nu_strong_times)
np.savetxt("numba_weak.txt", nu_weak_times)

speedups = speedup(nu_strong_times)
efficiencies = efficiency(speedups,t)

 17%|█▋        | 1/6 [00:00<00:02,  1.70it/s]

Numba parallel Lorentzian histogram: 0.59 seconds


 50%|█████     | 3/6 [00:01<00:00,  3.35it/s]

Numba parallel Lorentzian histogram: 0.30 seconds
Numba parallel Lorentzian histogram: 0.15 seconds


100%|██████████| 6/6 [00:01<00:00,  4.66it/s]

...

Numba parallel Lorentzian histogram: 0.58 seconds


 75%|███████▌  | 3/4 [00:01<00:00,  1.68it/s]

Numba parallel Lorentzian histogram: 0.61 seconds


100%|██████████| 4/4 [00:02<00:00,  1.67it/s]

Numba parallel Lorentzian histogram: 0.62 seconds


In [72]:
n = 10**7
bins = 100

counts = run_dask(n, n_counts=32, bins=bins)
plot_scaling(counts, technique="Numba", n=n, bins=bins)

### (g) Joblib

In [73]:
import joblib_lorentz
joblib_lorentz = il.reload(joblib_lorentz)
from joblib_lorentz import run_joblib

In [22]:
n = 10**7
bins = 1000

jb_strong_times = strong_scaling(run_joblib, n=n, bins=bins, technique='Joblib')
jb_weak_times = weak_scaling(run_joblib, n_per_div=n, bins=bins, technique='Joblib')


np.savetxt("joblib_strong.txt", jb_strong_times)
np.savetxt("joblib_weak.txt", jb_weak_times)

speedups = speedup(jb_strong_times)
efficiencies = efficiency(speedups,t)

100%|██████████| 6/6 [00:03<00:00,  1.73it/s]


100%|██████████| 4/4 [00:02<00:00,  1.55it/s]


In [74]:
n = 10**7
bins = 100

counts = run_joblib(n, n_counts=32, bins=bins)
plot_scaling(counts, technique="Joblib", n=n, bins=bins)

### (h) Mpire

In [75]:
import mpire_lorentz
mpire_lorentz = il.reload(mpire_lorentz)
from mpire_lorentz import run_mpire

In [28]:
n = 10**7
bins = 1000

jb_strong_times = strong_scaling(run_mpire, n=n, bins=bins, technique='Mpire')
jb_weak_times = weak_scaling(run_mpire, n_per_div=n, bins=bins, technique='Mpire')

np.savetxt("mpire_strong.txt", jb_strong_times)
np.savetxt("mpire_weak.txt", jb_weak_times)

speedups = speedup(jb_strong_times)
efficiencies = efficiency(speedups,t)

  0%|          | 0/6 [00:00<?, ?it/s]

100%|██████████| 6/6 [00:02<00:00,  2.09it/s]


100%|██████████| 4/4 [00:01<00:00,  2.04it/s]


In [77]:
n = 10**7
bins = 100

counts = run_mpire(n, n_counts=32, bins=bins)
plot_scaling(counts, technique="Mpire", n=n, bins=bins)

### (f) Mpi4py

In [30]:
# Performed through slurm job

### Fitting Strong Scaling Speedups

In [11]:

import numpy as np

def amdahl(p, f):
    return 1/(f + (1-f)/p)


In [13]:
chunks = [1,2,4,8,16,32]

methods = ['threading', 'ppe', 'numba', 'multiprocessing', 'mpire', 'mpi', 'dask', 'async', 'joblib']
m_name = ['Threading', 'ProcessPoolExecutor', 'Numba', 'Multiprocessing', 'Mpire', 'MPI', 'Dask', 'Asyncio', 'Joblib']
for m, name in zip(methods, m_name):
    strong_times = np.loadtxt(f"src/outputs/{m}_strong.txt")    
    speedups = speedup(strong_times)
    
    params, _ = curve_fit(amdahl, chunks, speedups)
    f = params[0]
    
    print(f"Estimated serial fraction for {name}: {f}")

Estimated serial fraction for Threading: 0.22944964901090306
Estimated serial fraction for ProcessPoolExecutor: 0.5392841044512618
Estimated serial fraction for Numba: 0.09232997391832067
Estimated serial fraction for Multiprocessing: 0.5439868998712462
Estimated serial fraction for Mpire: 0.9832222615364051
Estimated serial fraction for MPI: 0.040294945910714024
Estimated serial fraction for Dask: 0.21673366530524646
Estimated serial fraction for Asyncio: 0.8798056180344198
Estimated serial fraction for Joblib: 1.8270397483646366


### Check for 1e9 Case - Fastest Method -> MPI

In [14]:
strong_times = np.loadtxt(f"src/outputs/mpi_strong_1e9.txt")
speedups = speedup(strong_times)
params, _ = curve_fit(amdahl, chunks, speedups)
f = params[0]

print(f"Estimated serial fraction for MPI 1e9 Case: {f}")

Estimated serial fraction for MPI 1e9 Case: 0.013658048903973347


In [15]:
print(strong_times)

[15.91138127  8.03574662  4.06012237  2.2875091   1.07903032  0.7226723 ]
